# Chapter 4: Models with Distance Metrics and Nearest Neighbors
## 📌 Summary
K-Nearest Neighbors (KNN) adalah algoritma **non-parametric** dan **lazy learning** yang membuat prediksi berdasarkan kedekatan dengan training examples.

## 🧠 Theoretical Deep-Dive

### 4.1 Distance Metrics
- **Euclidean**: √(Σ(xᵢ-yᵢ)²) → standar, untuk continuous features
- **Manhattan**: Σ|xᵢ-yᵢ| → lebih robust terhadap outlier
- **Minkowski**: (Σ|xᵢ-yᵢ|ᵖ)^(1/p) → generalisasi (p=1: Manhattan, p=2: Euclidean)
- **Cosine**: 1 - (x·y)/(|x||y|) → untuk text/high-dimensional data

### 4.2 K-Nearest Neighbors Classification
Untuk titik baru x, KNN:
1. Hitung jarak ke semua training points
2. Pilih k tetangga terdekat
3. Ambil majority vote (classification) atau rata-rata (regression)

**Trade-off k:**
- k kecil → low bias, high variance (overfitting)
- k besar → high bias, low variance (underfitting)

### 4.3 Ball Tree & KD-Tree
Struktur data untuk mempercepat pencarian tetangga:
- **KD-Tree**: efisien untuk dimensi rendah (<20)
- **Ball Tree**: lebih efisien untuk dimensi tinggi atau non-Euclidean metrics

### 4.4 Weighted KNN
Tetangga yang lebih dekat diberi bobot lebih besar: weight = 1/distance

## 💻 Code Reproduction

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

knn = KNeighborsClassifier(n_neighbors=5, metric="euclidean")
knn.fit(X_train_s, y_train)

print("Accuracy:", knn.score(X_test_s, y_test))
print("\nClassification Report:")
print(classification_report(y_test, knn.predict(X_test_s), target_names=["setosa","versicolor","virginica"]))

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
import numpy as np
import matplotlib.pyplot as plt

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

k_range = range(1, 31)
scores = []
for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    cv_scores = cross_val_score(knn, X_train_s, y_train, cv=5)
    scores.append(cv_scores.mean())

print("Best k:", k_range[np.argmax(scores)], "with CV accuracy:", max(scores).round(4))

plt.figure(figsize=(8, 4))
plt.plot(k_range, scores, marker="o")
plt.xlabel("k"); plt.ylabel("CV Accuracy")
plt.title("KNN: Effect of k")
plt.grid(True); plt.tight_layout(); plt.show()

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

X, y = make_regression(n_samples=200, n_features=5, noise=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

knn_reg = KNeighborsRegressor(n_neighbors=5, weights="distance")
knn_reg.fit(X_train_s, y_train)
y_pred = knn_reg.predict(X_test_s)

print("MSE:", mean_squared_error(y_test, y_pred).round(2))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)).round(2))
print("R²:", r2_score(y_test, y_pred).round(4))

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X, y = make_classification(n_samples=300, n_features=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# Compare distance metrics and algorithms
for metric in ["euclidean", "manhattan", "minkowski"]:
    for algo in ["ball_tree", "kd_tree"]:
        if metric == "minkowski" and algo == "kd_tree":
            continue  # minkowski only with ball_tree for p!=1,2
        knn = KNeighborsClassifier(n_neighbors=5, metric=metric, algorithm=algo)
        knn.fit(X_train_s, y_train)
        acc = knn.score(X_test_s, y_test)
        print(f"metric={metric:12} algo={algo:10} acc={acc:.4f}")

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# Uniform vs Distance weighting
for weight in ["uniform", "distance"]:
    knn = KNeighborsClassifier(n_neighbors=5, weights=weight)
    knn.fit(X_train_s, y_train)
    print(f"weights={weight:10}: accuracy = {knn.score(X_test_s, y_test):.4f}")

# Get distances to neighbors
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train_s, y_train)
dists, indices = knn.kneighbors(X_test_s[:3])
print("\nDistances to 3 nearest neighbors (first 3 test samples):")
print(dists.round(3))
print("Indices:", indices)

## ✅ Chapter Summary
- KNN adalah **lazy learner**: tidak ada training phase, semua komputasi saat prediksi
- **Selalu scale** fitur sebelum KNN (sangat sensitif terhadap skala)
- k optimal biasanya dicari via **cross-validation**
- `weights='distance'` sering lebih baik dari `'uniform'`
- **Ball Tree** > KD-Tree untuk data high-dimensional atau non-Euclidean metrics
- KNN O(n) per prediksi → lambat untuk dataset besar